# SOTA CPT Eval + Merge v2 (Notebook C_sota)

Evaluates the SOTA adapter with decision-grade metrics (plan Phase 3):

1. Multi-bucket PPL for **base / Phase-1 adapter / v2 adapter** (`EVAL_BASE=True` default)
2. Deterministic **greedy** style/doctrine/forgetting probes (+ optional sampled flavor)
3. **Catechism MCQ** log-likelihood (WSC absorption + Heidelberg generalization)
4. Merge/GGUF only after success criteria pass (`RUN_MERGE = False` until then)

Flagship base: `unsloth/Qwen3.5-4B-Base`

**Does not replace** `C_eval_and_merge.ipynb`.


## 1. Install

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Config

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "unsloth/Qwen3.5-4B-Base"  # must match training base
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

ADAPTER_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-lora/theology_cpt_lora"
# Optional Phase-1 adapter for honest comparison (same base only — else use %Δ vs own base)
PHASE1_ADAPTER_PATH = None  # e.g. "/kaggle/input/.../spurgeon_phase1_lora"
HOLDOUT_ROOT = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-dataset/theology_holdouts"
MCQ_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-corpus/catechism_mcq.json"

EVAL_BASE = True
EVAL_PHASE1 = False  # set True if PHASE1_ADAPTER_PATH is same base family
MAX_DOCS_PER_BUCKET = 50
PROBE_SEED = 42
RUN_MERGE = False  # set True only after §5 success criteria pass

OUT_LORA = "/kaggle/working/theology_cpt_lora_final"
OUT_MERGED = "/kaggle/working/theology_cpt_merged_hf"
OUT_METRICS = "/kaggle/working/theology_cpt_eval_metrics.json"

print("Config OK")

## 3. Load v2 adapter

In [ ]:
from unsloth import FastLanguageModel
import torch
import json
import math
from datasets import load_from_disk

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)
print("Loaded adapter from", ADAPTER_PATH)

## 4. Multi-holdout PPL + Δ table

In [ ]:
def eval_ppl(model, tokenizer, dataset, max_docs=None, max_seq=MAX_SEQ_LENGTH):
    total_loss = 0.0
    total_tokens = 0
    n = len(dataset) if max_docs is None else min(len(dataset), max_docs)
    for i in range(n):
        text = dataset[i]["text"]
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        num_tokens = inputs["input_ids"].size(1)
        if num_tokens > max_seq:
            inputs = {k: v[:, :max_seq] for k, v in inputs.items()}
            num_tokens = max_seq
        if num_tokens < 2:
            continue
        with torch.no_grad():
            out = model(**inputs, labels=inputs["input_ids"])
            loss = out.loss.item()
        total_loss += loss * num_tokens
        total_tokens += num_tokens
    if total_tokens == 0:
        return {"tokens": 0, "loss": None, "ppl": None, "docs": 0}
    avg = total_loss / total_tokens
    return {"tokens": total_tokens, "loss": avg, "ppl": math.exp(avg), "docs": n}

buckets = ["spurgeon", "puritan", "confession", "general"]
metrics = {"v2": {}, "base": {}, "phase1": {}, "delta_vs_base_pct": {}}

for name in buckets:
    path = os.path.join(HOLDOUT_ROOT, name)
    if not os.path.exists(path):
        print("skip missing", path)
        continue
    ds = load_from_disk(path)
    print(f"Evaluating v2 PPL on {name} ({len(ds)} docs)...")
    metrics["v2"][name] = eval_ppl(model, tokenizer, ds, max_docs=MAX_DOCS_PER_BUCKET)
    m = metrics["v2"][name]
    if m["ppl"] is not None:
        print(f"  v2 {name}: ppl={m['ppl']:.2f} loss={m['loss']:.4f} tokens={m['tokens']:,}")

def score_model(label, model_name_or_path):
    print(f"Loading {label} from {model_name_or_path}...")
    m, t = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(m)
    out = {}
    for name in buckets:
        path = os.path.join(HOLDOUT_ROOT, name)
        if not os.path.exists(path):
            continue
        ds = load_from_disk(path)
        out[name] = eval_ppl(m, t, ds, max_docs=MAX_DOCS_PER_BUCKET)
        if out[name]["ppl"] is not None:
            print(f"  {label} {name}: ppl={out[name]['ppl']:.2f}")
    del m
    torch.cuda.empty_cache()
    return out

if EVAL_BASE:
    metrics["base"] = score_model("base", MODEL_NAME)
if EVAL_PHASE1 and PHASE1_ADAPTER_PATH:
    metrics["phase1"] = score_model("phase1", PHASE1_ADAPTER_PATH)

# Δ table vs base
print("\n=== Δ PPL vs base (% lower is better absorption for domain buckets) ===")
print(f"{'bucket':12s}  {'base':>8s}  {'v2':>8s}  {'%Δ':>8s}")
for name in buckets:
    b = (metrics["base"].get(name) or {}).get("ppl")
    v = (metrics["v2"].get(name) or {}).get("ppl")
    if b and v:
        pct = 100.0 * (v - b) / b
        metrics["delta_vs_base_pct"][name] = round(pct, 2)
        print(f"{name:12s}  {b:8.2f}  {v:8.2f}  {pct:7.1f}%")

with open(OUT_METRICS, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("Wrote", OUT_METRICS)

## 5. Probes (greedy = comparable; sample = flavor)

In [ ]:
style_prompts = [
    "The love of Christ is not a cold, speculative thing. It is",
    "Text: Romans 8:28. 'And we know that all things work together for good to them that love God.' My dear friends,",
    "What, then, is saving faith? Let us examine this question carefully, for",
]
doctrine_prompts = [
    "The Westminster Confession teaches that God from all eternity did,",
    "Justification is an act of God's free grace wherein He",
    "True saving faith rests upon Christ alone, for",
]
forgetting_prompts = [
    "The capital of France is",
    "Photosynthesis in green plants converts light energy into",
    "In the nineteenth century, the Industrial Revolution",
]

def generate(prompt, max_new_tokens=150, greedy=True):
    torch.manual_seed(PROBE_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(PROBE_SEED)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    gen_kwargs = dict(max_new_tokens=max_new_tokens)
    if greedy:
        gen_kwargs.update(dict(do_sample=False))
    else:
        gen_kwargs.update(dict(do_sample=True, temperature=0.7, top_p=0.9))
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)

probe_log = {"greedy": {}, "sampled": {}}
for label, prompts in [
    ("style", style_prompts),
    ("doctrine", doctrine_prompts),
    ("forgetting", forgetting_prompts),
]:
    probe_log["greedy"][label] = []
    print(f"\n=== {label.upper()} (greedy, seed={PROBE_SEED}) ===")
    for p in prompts:
        text = generate(p, greedy=True, max_new_tokens=120 if label != "forgetting" else 40)
        print("\n---\n", text[:800])
        probe_log["greedy"][label].append({"prompt": p, "completion": text})

# Optional flavor (non-comparable)
print("\n=== Style (sampled, flavor only) ===")
probe_log["sampled"]["style"] = []
for p in style_prompts[:1]:
    text = generate(p, greedy=False)
    print(text[:500])
    probe_log["sampled"]["style"].append({"prompt": p, "completion": text})

metrics["probes"] = probe_log
with open(OUT_METRICS, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

## 6. Catechism MCQ log-likelihood (WSC + Heidelberg)

In [ ]:
def option_logprob(model, tok, prompt, option):
    full = tok(prompt + " " + option, return_tensors="pt").to("cuda")
    p_len = tok(prompt, return_tensors="pt")["input_ids"].size(1)
    with torch.no_grad():
        logits = model(**full).logits[:, :-1].float()
    ids = full["input_ids"][:, 1:]
    lp = torch.log_softmax(logits, -1).gather(-1, ids.unsqueeze(-1)).squeeze(-1)
    # mean logprob of option tokens only
    start = max(0, p_len - 1)
    if start >= lp.size(1):
        return float("-inf")
    return lp[0, start:].mean().item()

def mcq_accuracy(model, tok, items):
    if not items:
        return None
    hits = 0
    for it in items:
        opts = [it["a"], *it.get("distractors", [])]
        scores = [option_logprob(model, tok, f"Q. {it['q']}\nA.", o) for o in opts]
        hits += int(scores.index(max(scores)) == 0)
    return hits / len(items)

mcq_metrics = {}
if os.path.exists(MCQ_PATH):
    mcq = json.loads(open(MCQ_PATH, encoding="utf-8").read())
    sets = mcq.get("sets") or mcq
    for set_name, items in sets.items():
        if not items:
            continue
        acc = mcq_accuracy(model, tokenizer, items)
        mcq_metrics[f"v2_{set_name}"] = acc
        print(f"MCQ v2 {set_name}: {acc:.1%} (n={len(items)})")
    if EVAL_BASE and any(sets.values()):
        base_m, base_t = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=LOAD_IN_4BIT,
        )
        FastLanguageModel.for_inference(base_m)
        for set_name, items in sets.items():
            if not items:
                continue
            acc = mcq_accuracy(base_m, base_t, items)
            mcq_metrics[f"base_{set_name}"] = acc
            print(f"MCQ base {set_name}: {acc:.1%}")
        del base_m
        torch.cuda.empty_cache()
else:
    print(
        f"NOTE: no MCQ file at {MCQ_PATH}. Build with "
        "scripts/09_build_catechism_mcq.py and upload to corpus dataset."
    )

metrics["mcq"] = mcq_metrics
with open(OUT_METRICS, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("Updated", OUT_METRICS)

## 7. Save LoRA + optional merge (gated)

In [ ]:
print("Saving final LoRA to", OUT_LORA)
model.save_pretrained(OUT_LORA)
tokenizer.save_pretrained(OUT_LORA)

if not RUN_MERGE:
    print("RUN_MERGE=False — skip merge/GGUF until §5 success criteria pass.")
else:
    try:
        print("Merging 16-bit HF weights to", OUT_MERGED)
        model.save_pretrained_merged(OUT_MERGED, tokenizer, save_method="merged_16bit")
        print("Merge complete")
        print(
            "GGUF note: if exporting to Ollama, load tokenizer FROM the merged folder "
            "(vocab-shift fix — see project memory bugs/ollama-tokenizer-corruption-fix)."
        )
    except Exception as e:
        print("Merge skipped/failed:", e)

print("Done. Metrics at", OUT_METRICS)

## Success criteria (checklist)

| Probe | Target |
|-------|--------|
| Manifest | ≥4 buckets, shares in range, holdouts non-empty, `verified_tokens` present |
| Spurgeon PPL | Better than base; after base swap use %Δ-vs-own-base + probes (not raw PPL vs Phase-1) |
| Puritan / confession PPL | ≥15% better than base |
| General PPL | ≤10% worse than base |
| Heidelberg MCQ | ≥ +10 points absolute vs base |
| WSC MCQ | Near-ceiling (absorption) |
| Greedy style | Preferable Spurgeon-ness vs base |
